# Notebook 10: Reasoning via RL -- DeepSeek-R1 and GRPO

**Critical Notebook** -- Reasoning models are the hottest topic in frontier AI (2024-2025).

**Runtime**: ~30-45 min on Colab T4  
**GPU Memory**: ~6-8 GB peak  
**Objective**: Understand and implement GRPO, train a small model to reason about math problems, and observe emergent chain-of-thought behavior.

---
## 1. Self-Quiz: Active Recall (Answer Before Reading On)

Try answering these from memory before proceeding:

1. **How does DeepSeek-R1 differ from GPT-4?** What is the fundamental training paradigm difference?
2. **What is GRPO?** How does it differ from standard PPO?
3. **Why no critic network?** What replaces the value function in GRPO?
4. **What are verifiable rewards?** Why are they important for reasoning RL?
5. **What is test-time compute scaling?** How does it differ from model scaling?
6. **What is the difference between a Process Reward Model (PRM) and an Outcome Reward Model (ORM)?**
7. **What did DeepSeek-R1-Zero demonstrate?** Why was it surprising?

<details>
<summary>Click to reveal key points</summary>

1. GPT-4 was trained with SFT + RLHF on human preferences. DeepSeek-R1 uses RL with verifiable rewards (correctness on math/code) to learn reasoning strategies, not just mimic human responses.
2. GRPO = Group Relative Policy Optimization. Instead of learning a value function (critic), it samples G completions per prompt and uses group-normalized rewards as advantages.
3. The critic is replaced by within-group normalization: advantage_i = (r_i - mean(r)) / std(r). This saves ~50% memory and simplifies training.
4. Verifiable rewards are rewards that can be checked automatically (e.g., math answer correctness, code test passing). They provide reliable training signal without human annotation.
5. Test-time compute scaling: instead of training a bigger model, spend more compute at inference (generate many candidates, use verifier to pick best). New scaling axis.
6. ORM rewards only the final answer. PRM rewards each intermediate step. PRMs are more effective but harder to create training data for.
7. DeepSeek-R1-Zero showed that pure RL (without any SFT on reasoning traces) can produce emergent chain-of-thought reasoning, self-correction, and "aha moments."
</details>

---
## 2. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets matplotlib seaborn pandas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import numpy as np
import re
import json
import time
import copy
import warnings
import gc
from collections import defaultdict
from typing import List, Tuple, Optional, Dict

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="colorblind")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Model selection
if device == "cuda" and torch.cuda.get_device_properties(0).total_memory > 12e9:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    print("Using TinyLlama-1.1B (RunPod mode)")
else:
    MODEL_NAME = "gpt2"
    print("Using GPT-2 (Colab mode)")

---
## 3. The Reasoning Paradigm Shift

### Before: Reasoning via Prompting

For years, the dominant approach to LLM reasoning was **prompting**:

- **Chain-of-Thought (CoT)**: "Let's think step by step" (Wei et al., 2022)
- **Few-shot examples**: Show the model worked examples of reasoning
- **Self-consistency**: Generate multiple CoT paths, take majority vote (Wang et al., 2022)
- **Tree-of-Thought**: Explore multiple reasoning branches systematically (Yao et al., 2023)

The key limitation: **the model does not actually learn to reason better**. It is just prompted to format its output like reasoning. The underlying model is unchanged.

### After: Reasoning via RL Training

The paradigm shift of 2024-2025:

| Aspect | Prompting Approach | RL Training Approach |
|--------|-------------------|---------------------|
| Model changes? | No -- same weights | Yes -- weights updated |
| Reasoning quality | Limited by base model | Improves with training |
| Compute cost | Inference only | Training + inference |
| Generalization | Depends on prompt | Transfers across tasks |
| Self-correction | Rare, unreliable | Emerges from training |

### Key Insight from DeepSeek-R1-Zero

The **most surprising finding** in DeepSeek-R1 (Jan 2025):

> Pure RL training -- without ANY supervised fine-tuning on reasoning traces -- produces emergent chain-of-thought reasoning, including self-correction ("Wait, let me reconsider...") and verification ("Let me check this answer...").

This means reasoning is not just a format trick -- **RL discovers reasoning as a strategy for maximizing reward**. The model learns that thinking step-by-step leads to more correct answers, which leads to higher reward.

### The Landscape

| Model | Organization | Approach | Key Innovation |
|-------|-------------|----------|----------------|
| o1 / o3 | OpenAI | RL on reasoning | Test-time compute scaling, PRMs (likely) |
| DeepSeek-R1 | DeepSeek | GRPO + verifiable rewards | Open-source, emergent reasoning from pure RL |
| Gemini 2.0 Flash Thinking | Google DeepMind | RL on reasoning | Efficient reasoning, thinking tokens |

*(2024-2025-era snapshot; see notebook 23 for the current landscape.)*

**This is arguably the biggest ML breakthrough of 2024-2025.** It shows that RL can elicit capabilities that SFT cannot.

---
## 4. GRPO: Group Relative Policy Optimization

### From the DeepSeekMath Paper

**Paper:** [DeepSeekMath: Pushing the Limits of Mathematical Reasoning](https://arxiv.org/abs/2402.03300) (Shao et al., Feb 2024)

### The Problem with PPO

Standard PPO for LLM alignment requires:
1. **Policy model** (the LLM being trained)
2. **Reference model** (frozen copy for KL penalty)
3. **Reward model** (trained on human preferences)
4. **Value/critic model** (estimates expected future reward)

That is **four large models** in memory simultaneously. For a 7B model, this requires ~112 GB just for model weights (4 x 7B x 4 bytes).

### GRPO's Key Innovation: Eliminate the Critic

GRPO replaces the learned value function with **group normalization**:

```
Standard PPO:         advantage_i = reward_i - V(state_i)         # requires learned V
GRPO:                 advantage_i = (r_i - mean(r_group)) / std(r_group)  # just statistics
```

### Algorithm (Pseudocode)

```
for each training step:
    for each prompt q in batch:
        1. Sample G completions: {o_1, ..., o_G} ~ pi_theta(.|q)
        2. Compute reward for each: {r_1, ..., r_G}
        3. Normalize within group: 
           advantage_i = (r_i - mean(r)) / std(r)
        4. For each completion o_i:
           ratio = pi_theta(o_i|q) / pi_old(o_i|q)
           L_clip = min(ratio * A_i, clip(ratio, 1-eps, 1+eps) * A_i)
           L_kl = beta * KL(pi_theta || pi_ref)
           Loss += -L_clip + L_kl
    Update theta to minimize Loss
```

### Why This Works

1. **Within-group normalization is a good baseline**: If you sample enough completions (G >= 4), the group mean is a reasonable estimate of the value.
2. **No learned value function to overfit**: The critic in PPO can overfit or lag behind the policy, causing instability.
3. **Simpler implementation**: Fewer moving parts means fewer bugs and easier debugging.
4. **Memory savings**: ~50% reduction (no critic model).

The tradeoff: GRPO needs to generate G samples per prompt (vs 1 for PPO), which increases inference cost during training. But this is often cheaper than storing and training a value network.

In [ ]:
# GRPO Implementation (~60 lines of clean, annotated code)

class GRPO:
    """Group Relative Policy Optimization.
    
    Key differences from PPO:
    - No critic/value network
    - Advantages computed via group normalization
    - Samples G completions per prompt
    """
    
    def __init__(
        self,
        model: AutoModelForCausalLM,
        ref_model: AutoModelForCausalLM,
        tokenizer: AutoTokenizer,
        group_size: int = 8,
        clip_eps: float = 0.2,
        kl_coeff: float = 0.05,
        max_new_tokens: int = 128,
    ):
        self.model = model
        self.ref_model = ref_model  # frozen reference for KL penalty
        self.tokenizer = tokenizer
        self.group_size = group_size  # G: number of completions per prompt
        self.clip_eps = clip_eps
        self.kl_coeff = kl_coeff
        self.max_new_tokens = max_new_tokens
        self.device = next(model.parameters()).device
    
    @torch.no_grad()
    def sample_group(self, prompt: str) -> List[str]:
        """Sample G completions from the current policy for a given prompt."""
        inputs = self.tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=256
        ).to(self.device)
        
        responses = []
        for _ in range(self.group_size):
            output = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=1.0,  # full temperature for diversity
                top_p=0.95,
                pad_token_id=self.tokenizer.pad_token_id,
            )
            response = self.tokenizer.decode(
                output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
            )
            responses.append(response)
        return responses
    
    def compute_rewards(
        self, prompt: str, responses: List[str], gold_answer: str
    ) -> List[float]:
        """Compute rewards for each response. 
        
        For math: binary correctness (1.0 if answer matches, 0.0 otherwise)
        Plus a small format bonus for showing work.
        """
        rewards = []
        for response in responses:
            reward = 0.0
            
            # Correctness reward: extract final numerical answer
            extracted = self._extract_answer(response)
            if extracted is not None and self._answers_match(extracted, gold_answer):
                reward += 1.0  # Correct answer
            
            # Format reward: small bonus for showing reasoning steps
            if any(marker in response.lower() for marker in 
                   ["step", "first", "then", "therefore", "so ", "because", "="]):
                reward += 0.1  # Shows some reasoning
            
            rewards.append(reward)
        return rewards
    
    def compute_group_advantages(self, rewards: List[float]) -> List[float]:
        """Normalize rewards within the group to get advantages.
        
        This is the KEY innovation of GRPO:
        advantage_i = (r_i - mean(r)) / (std(r) + epsilon)
        
        No learned value function needed!
        """
        rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
        mean_r = rewards_tensor.mean()
        std_r = rewards_tensor.std() + 1e-8  # avoid division by zero
        advantages = ((rewards_tensor - mean_r) / std_r).tolist()
        return advantages
    
    def compute_log_probs(
        self, model: AutoModelForCausalLM, prompt: str, response: str
    ) -> torch.Tensor:
        """Compute log probabilities of response tokens given prompt."""
        full_text = prompt + response
        inputs = self.tokenizer(
            full_text, return_tensors="pt", truncation=True, max_length=384
        ).to(self.device)
        
        prompt_len = self.tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=256
        ).input_ids.shape[1]
        
        with torch.no_grad() if model is self.ref_model else torch.enable_grad():
            outputs = model(**inputs)
        
        # Get log probs for response tokens only
        logits = outputs.logits[:, prompt_len-1:-1, :]  # shifted
        labels = inputs.input_ids[:, prompt_len:]  # response tokens
        
        # Trim to match lengths
        min_len = min(logits.shape[1], labels.shape[1])
        if min_len == 0:
            return torch.tensor(0.0, device=self.device)
        logits = logits[:, :min_len, :]
        labels = labels[:, :min_len]
        
        log_probs = F.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
        return token_log_probs.sum()  # sum over response tokens
    
    @torch.no_grad()
    def compute_old_log_probs(self, prompt: str, responses: List[str]) -> List[torch.Tensor]:
        """Compute log probs for all responses under the current policy.
        
        Call this BEFORE any gradient updates to get pi_old log probs.
        These are used in the importance ratio: pi_theta / pi_old.
        """
        old_lps = []
        for response in responses:
            if len(response.strip()) == 0:
                old_lps.append(torch.tensor(0.0, device=self.device))
            else:
                lp = self.compute_log_probs(self.model, prompt, response)
                old_lps.append(lp.detach())
        return old_lps

    def update(
        self,
        prompt: str,
        responses: List[str],
        advantages: List[float],
        optimizer: torch.optim.Optimizer,
        old_log_probs: Optional[List[torch.Tensor]] = None,
    ) -> Dict[str, float]:
        """Perform a GRPO update step.
        
        Uses clipped surrogate objective (like PPO) + KL penalty.
        
        Args:
            old_log_probs: Log probs computed BEFORE gradient updates (pi_old).
                           If None, computes them now (only correct if no prior updates).
        """
        if old_log_probs is None:
            old_log_probs = self.compute_old_log_probs(prompt, responses)
        
        total_loss = torch.tensor(0.0, device=self.device)
        total_clip_loss = 0.0
        total_kl = 0.0
        n_valid = 0
        
        for idx, (response, advantage) in enumerate(zip(responses, advantages)):
            if len(response.strip()) == 0:
                continue
            
            # Current policy log prob (with gradients for backprop)
            curr_log_prob = self.compute_log_probs(self.model, prompt, response)
            
            # Old policy log prob (computed at sample time, before gradient updates)
            # NOTE: We use stored old_log_probs passed in from sample time.
            # Computing old_log_prob from self.model here would give ratio=1 always
            # (same model, same weights), making the clipping ineffective.
            old_log_prob = old_log_probs[idx]
            
            # Reference policy log prob (for KL)
            with torch.no_grad():
                ref_log_prob = self.compute_log_probs(self.ref_model, prompt, response)
            
            # Importance ratio
            ratio = torch.exp(curr_log_prob - old_log_prob)
            
            # Clipped surrogate objective
            adv_tensor = torch.tensor(advantage, device=self.device)
            unclipped = ratio * adv_tensor
            clipped = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * adv_tensor
            clip_loss = -torch.min(unclipped, clipped)
            
            # KL penalty
            # NOTE: this is the k1 estimator (log pi - log pi_ref), applied at the
            # SEQUENCE level. The GRPO paper instead uses the unbiased, non-negative
            # k3 estimator PER TOKEN: exp(ref-cur) - (ref-cur) - 1. This sequence-level
            # k1 version is a simplification for the demo.
            kl = curr_log_prob - ref_log_prob  # approximate KL (k1 estimator)
            
            # Combined loss
            loss = clip_loss + self.kl_coeff * kl
            total_loss = total_loss + loss
            
            total_clip_loss += clip_loss.item()
            total_kl += kl.item()
            n_valid += 1
        
        if n_valid > 0:
            total_loss = total_loss / n_valid
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()
        
        return {
            "loss": total_loss.item() if n_valid > 0 else 0.0,
            "clip_loss": total_clip_loss / max(n_valid, 1),
            "kl": total_kl / max(n_valid, 1),
        }
    
    @staticmethod
    def _extract_answer(text: str) -> Optional[str]:
        """Extract the final numerical answer from a response."""
        # Try common patterns: "The answer is X", "= X", "####X"
        patterns = [
            r"(?:the answer is|answer:?)\s*[\$]?\s*([\-]?[\d,]+\.?\d*)",
            r"####\s*([\-]?[\d,]+\.?\d*)",
            r"=\s*[\$]?\s*([\-]?[\d,]+\.?\d*)\s*$",
            r"([\-]?[\d,]+\.?\d*)\s*$",  # last number in response
        ]
        for pattern in patterns:
            match = re.search(pattern, text.lower().strip())
            if match:
                return match.group(1).replace(",", "")
        return None
    
    @staticmethod
    def _answers_match(predicted: str, gold: str) -> bool:
        """Check if predicted answer matches gold answer."""
        try:
            pred_num = float(predicted)
            gold_num = float(gold.replace(",", ""))
            return abs(pred_num - gold_num) < 1e-2
        except (ValueError, TypeError):
            return predicted.strip() == gold.strip()

In [ ]:
# Side-by-side comparison: GRPO vs PPO

comparison_text = """
============================================================
GRPO vs PPO: Side-by-Side Comparison
============================================================

PPO (Standard LLM Alignment):
  Models needed:   4 (policy, reference, reward, critic)
  Advantage:       A_t = r_t + gamma*V(s_{t+1}) - V(s_t)      # learned value function
  Memory:          ~4x model size
  Complexity:      High (GAE, value loss, reward normalization)
  Samples/prompt:  1 (or few)

GRPO (DeepSeek's Innovation):
  Models needed:   2 (policy, reference) + reward function
  Advantage:       A_i = (r_i - mean(r_group)) / std(r_group)  # group statistics
  Memory:          ~2x model size
  Complexity:      Low (just normalization)
  Samples/prompt:  G (typically 4-16)

What is DIFFERENT:
  - No critic network (replaced by group normalization)
  - No GAE (Generalized Advantage Estimation)
  - Multiple samples per prompt (enables group normalization)

What is the SAME:
  - Clipped surrogate objective (PPO-clip)
  - KL penalty against reference policy
  - Policy gradient framework
  - Gradient clipping, learning rate scheduling
============================================================
"""
print(comparison_text)

---
## 5. Training on Math Reasoning

In [ ]:
# Load GSM8K dataset (Grade School Math)
print("Loading GSM8K dataset...")
gsm8k = load_dataset("openai/gsm8k", "main")
print(f"Train: {len(gsm8k['train'])} examples")
print(f"Test: {len(gsm8k['test'])} examples")

# Examine format
example = gsm8k["train"][0]
print(f"\nExample question:\n{example['question']}")
print(f"\nExample answer:\n{example['answer']}")

In [ ]:
def extract_gsm8k_answer(answer_text: str) -> str:
    """Extract the final numerical answer from GSM8K format.
    
    GSM8K uses #### to mark the final answer.
    Example: '...The answer is 42.\n#### 42'
    """
    match = re.search(r"####\s*([\-]?[\d,]+\.?\d*)", answer_text)
    if match:
        return match.group(1).replace(",", "")
    return ""


def format_math_prompt(question: str) -> str:
    """Format a math question as a prompt for the model."""
    return f"Question: {question}\nLet me solve this step by step.\n"


# Prepare training subset
N_TRAIN_MATH = 200  # Small subset for demo
N_EVAL_MATH = 50

train_questions = []
train_answers = []
for i in range(N_TRAIN_MATH):
    q = gsm8k["train"][i]["question"]
    a = extract_gsm8k_answer(gsm8k["train"][i]["answer"])
    if a:  # only include if we can extract the answer
        train_questions.append(q)
        train_answers.append(a)

eval_questions = []
eval_answers = []
for i in range(N_EVAL_MATH):
    q = gsm8k["test"][i]["question"]
    a = extract_gsm8k_answer(gsm8k["test"][i]["answer"])
    if a:
        eval_questions.append(q)
        eval_answers.append(a)

print(f"Training questions: {len(train_questions)}")
print(f"Eval questions: {len(eval_questions)}")
print(f"\nSample: Q='{train_questions[0][:80]}...' A='{train_answers[0]}'")

In [ ]:
# Load model and create GRPO trainer
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Create GRPO instance
grpo = GRPO(
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    group_size=4,       # G=4 completions per prompt (reduced for speed)
    clip_eps=0.2,
    kl_coeff=0.05,
    max_new_tokens=100,  # shorter responses for speed
)

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
print("GRPO trainer initialized.")

In [ ]:
# Training loop
NUM_STEPS = 100  # Adjust based on available time
EVAL_EVERY = 20
LOG_EVERY = 5

# Tracking metrics
train_log = {
    "step": [],
    "loss": [],
    "kl": [],
    "avg_reward": [],
    "accuracy": [],
    "avg_response_length": [],
}

# Store example outputs at different training stages
example_outputs = {}  # step -> list of (prompt, response) pairs

print("=" * 60)
print("Starting GRPO Training on Math Reasoning")
print("=" * 60)

start_time = time.time()

for step in range(NUM_STEPS):
    # Sample a random training question
    idx = np.random.randint(len(train_questions))
    question = train_questions[idx]
    gold_answer = train_answers[idx]
    prompt = format_math_prompt(question)
    
    # GRPO step: sample group, compute rewards, compute advantages, update
    responses = grpo.sample_group(prompt)
    rewards = grpo.compute_rewards(prompt, responses, gold_answer)
    advantages = grpo.compute_group_advantages(rewards)
    # Compute pi_old log probs BEFORE gradient update (critical for correct ratio)
    old_log_probs = grpo.compute_old_log_probs(prompt, responses)
    step_metrics = grpo.update(prompt, responses, advantages, optimizer, old_log_probs)
    
    # Log metrics
    avg_reward = np.mean(rewards)
    avg_resp_len = np.mean([len(r.split()) for r in responses])
    
    if step % LOG_EVERY == 0:
        train_log["step"].append(step)
        train_log["loss"].append(step_metrics["loss"])
        train_log["kl"].append(step_metrics["kl"])
        train_log["avg_reward"].append(avg_reward)
        train_log["avg_response_length"].append(avg_resp_len)
    
    if step % LOG_EVERY == 0:
        print(f"Step {step:4d} | Loss: {step_metrics['loss']:.4f} | "
              f"KL: {step_metrics['kl']:.4f} | Avg Reward: {avg_reward:.2f} | "
              f"Avg Len: {avg_resp_len:.0f}")
    
    # Save example outputs at key checkpoints
    if step in [0, NUM_STEPS // 5, NUM_STEPS // 2, NUM_STEPS - 1]:
        example_outputs[step] = []
        for eq_idx in range(min(3, len(eval_questions))):
            eq = eval_questions[eq_idx]
            ep = format_math_prompt(eq)
            with torch.no_grad():
                inputs = tokenizer(ep, return_tensors="pt", truncation=True, max_length=256).to(device)
                output = model.generate(
                    **inputs, max_new_tokens=100, do_sample=True, 
                    temperature=0.7, pad_token_id=tokenizer.pad_token_id
                )
                resp = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            example_outputs[step].append((eq, resp))
    
    # Evaluate accuracy periodically
    if step % EVAL_EVERY == 0:
        model.eval()
        correct = 0
        total = min(20, len(eval_questions))  # Quick eval
        for ei in range(total):
            ep = format_math_prompt(eval_questions[ei])
            with torch.no_grad():
                inputs = tokenizer(ep, return_tensors="pt", truncation=True, max_length=256).to(device)
                output = model.generate(
                    **inputs, max_new_tokens=100, do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )
                resp = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            extracted = GRPO._extract_answer(resp)
            if extracted and GRPO._answers_match(extracted, eval_answers[ei]):
                correct += 1
        acc = correct / total
        train_log["accuracy"].append(acc)
        print(f"  >> Eval accuracy: {acc:.1%} ({correct}/{total})")
        model.train()

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.1f}s ({total_time/60:.1f} min)")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("GRPO Training on Math Reasoning (GSM8K)", fontsize=14, fontweight="bold")

# Loss curve
ax = axes[0, 0]
ax.plot(train_log["step"], train_log["loss"], color="#1f77b4", linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("GRPO Loss")

# Average reward
ax = axes[0, 1]
ax.plot(train_log["step"], train_log["avg_reward"], color="#2ca02c", linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Average Reward")
ax.set_title("Average Reward per Group")
ax.set_ylim(-0.1, 1.2)

# KL divergence
ax = axes[1, 0]
ax.plot(train_log["step"], train_log["kl"], color="#ff7f0e", linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("KL Divergence")
ax.set_title("KL from Reference Policy")

# Accuracy
ax = axes[1, 1]
acc_steps = [s for s in train_log["step"] if s % EVAL_EVERY == 0][:len(train_log["accuracy"])]
ax.plot(acc_steps, train_log["accuracy"], color="#d62728", linewidth=2, marker="o")
ax.set_xlabel("Training Step")
ax.set_ylabel("Eval Accuracy")
ax.set_title("Math Problem Accuracy")
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig("grpo_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved to grpo_training_curves.png")

---
## 7. Emergent Behaviors

This is the most fascinating part of the DeepSeek-R1-Zero finding: **the model develops reasoning traces WITHOUT being trained on chain-of-thought data**.

The only training signal is: "Did you get the right answer?" (binary reward)

Yet the model learns to:
1. Break problems into steps
2. Show intermediate calculations
3. Self-correct ("Wait, that does not seem right...")
4. Verify answers ("Let me check...")

These behaviors emerge because **reasoning is a strategy that increases the probability of getting the right answer**, and RL discovers strategies that maximize reward.

In [ ]:
# Show model outputs at different training stages
print("=" * 80)
print("EMERGENT REASONING: Model outputs across training")
print("=" * 80)

sorted_steps = sorted(example_outputs.keys())

for q_idx in range(min(3, len(eval_questions))):
    print(f"\n{'='*80}")
    print(f"QUESTION {q_idx + 1}: {eval_questions[q_idx]}")
    print(f"GOLD ANSWER: {eval_answers[q_idx]}")
    print(f"{'='*80}")
    
    for step in sorted_steps:
        if q_idx < len(example_outputs[step]):
            _, response = example_outputs[step][q_idx]
            # Truncate for display
            display = response[:400] + ("..." if len(response) > 400 else "")
            print(f"\n--- Step {step} ---")
            print(display)

print("\n" + "=" * 80)
print("ANALYSIS: What to look for:")
print("- Step 0: Random/incoherent output (no training yet)")
print("- Early steps: May start showing numbers and arithmetic")
print("- Later steps: May show step-by-step reasoning, intermediate results")
print("- Note: With a small model and limited training, emergent reasoning")
print("  may be modest. DeepSeek-R1 was built on DeepSeek-V3-Base (a 671B-parameter")
print("  MoE with 37B active) and much more training; the 7B scale you may recall is")
print("  DeepSeekMath-7B, from the GRPO paper.)")
print("=" * 80)

In [ ]:
# Analyze response characteristics over training
print("\nResponse Length Evolution:")
print("-" * 40)
for step in sorted_steps:
    lengths = [len(resp.split()) for _, resp in example_outputs[step]]
    avg_len = np.mean(lengths) if lengths else 0
    print(f"Step {step:4d}: avg {avg_len:.0f} words")

# Check for reasoning markers
reasoning_markers = ["step", "first", "then", "therefore", "so ", "because", "=", 
                     "calculate", "multiply", "add", "subtract", "total", "answer"]

print("\nReasoning Marker Frequency:")
print("-" * 40)
for step in sorted_steps:
    total_markers = 0
    for _, resp in example_outputs[step]:
        for marker in reasoning_markers:
            total_markers += resp.lower().count(marker)
    n_responses = max(len(example_outputs[step]), 1)
    print(f"Step {step:4d}: {total_markers / n_responses:.1f} markers per response")

---
## 8. Process vs Outcome Reward Models

### Outcome Reward Models (ORM)

An ORM evaluates only the **final answer**:

```
Input: "What is 15 + 27?"  
Response: "First, 15 + 20 = 35. Then 35 + 7 = 42. The answer is 42."  
ORM Score: 1.0 (correct final answer)
```

Problem: the model can get the right answer for the wrong reason, or use flawed reasoning that happens to reach the correct result.

### Process Reward Models (PRM)

A PRM evaluates **each reasoning step**:

```
Input: "What is 15 + 27?"  
Response: "First, 15 + 20 = 35."  -> PRM: correct step (1.0)  
          "Then 35 + 7 = 42."     -> PRM: correct step (1.0)  
          "The answer is 42."     -> PRM: correct conclusion (1.0)  
Overall PRM Score: 1.0
```

### "Let's Verify Step by Step" (Lightman et al., 2023)

**Paper:** [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050)

Key findings:
- PRMs significantly outperform ORMs on math reasoning
- The improvement is especially large when using best-of-N sampling (generate N solutions, select best according to reward model)
- PRM800K: a dataset of 800K step-level annotations for training PRMs
- **At best-of-1860 sampling on a MATH test subset, the PRM reaches 78.2%, vs 72.4% for an ORM and 69.6% for majority voting** (Lightman et al., 2023)

### Connection to o1 and Reasoning Models

While OpenAI has not disclosed o1's exact architecture, the reasoning model paradigm likely uses:
- **PRMs** to guide and evaluate reasoning chains
- **RL training** to teach the model when to think more vs conclude
- **Test-time compute scaling** to generate and verify multiple reasoning paths

### Why PRMs Are Hard to Build

1. **Annotation cost**: Labeling individual reasoning steps is much more expensive than labeling final answers
2. **Ambiguity**: Some steps are partially correct or correct in context but would be wrong in isolation
3. **Granularity**: How do you define a "step"? Sentence-level? Equation-level?
4. **Synthetic data**: Using models to generate PRM training data is promising but introduces distribution issues

### Interview Insight

"Process reward models are the key enabler for reasoning models. The challenge is obtaining step-level supervision at scale. I see three approaches: (1) human annotation (expensive but gold-standard), (2) synthetic generation using strong models (scalable but potentially biased), and (3) automatic verification for domains where steps can be checked programmatically (math proofs, code execution)."

---
## 9. Test-Time Compute Scaling

### The New Scaling Axis

Traditional scaling law: **bigger model = better performance** (Chinchilla, GPT-4)

New scaling axis: **more inference compute = better performance**

Instead of training a 10x bigger model, you can:
1. Generate N candidate solutions with a smaller model
2. Use a verifier (reward model or PRM) to select the best one
3. The quality scales with N (more candidates = higher chance of finding a good one)

### Optimal Test-Time Compute Allocation

**Paper:** [Scaling LLM Test-Time Compute Optimally](https://arxiv.org/abs/2408.03314) (Snell et al., 2024)

Key findings:

| Strategy | How It Works | When It Is Best |
|----------|-------------|----------------|
| Best-of-N | Generate N samples, pick best by reward | Harder problems (parallel sampling and PRM-guided beam search win here) |
| Sequential revision | Generate, critique, revise, repeat | Easier problems (revisions beat parallel sampling here) |
| Tree search | Explore multiple reasoning branches | Hard problems |
| Adaptive allocation | Spend more compute on harder problems | Mixed difficulty |

### When Is Test-Time Scaling Better Than Model Scaling?

From Snell et al.:
- For **easy problems**: test-time scaling is wasteful (the base model usually gets it right)
- For **medium problems**: test-time scaling is highly effective (more attempts = more likely to find the solution)
- For **very hard problems**: test-time scaling has diminishing returns (if the model fundamentally cannot solve it, more attempts will not help)
- The **sweet spot**: problems that the model can solve ~10-50% of the time with a single attempt

### Compute-Optimal Frontier

```
Performance
    |
    |          *  (large model, 1 sample)
    |        *
    |      * *  (small model, N samples + verifier)
    |    *
    |  *
    | *
    |*
    +-----------------------------------> Compute
```

At the same total compute budget, a smaller model with test-time scaling can match or exceed a larger model on many tasks.

### Practical Implications

1. **Deployment flexibility**: Use a smaller model for most queries, scale up compute for hard ones
2. **Cost optimization**: Test-time compute can be allocated dynamically based on problem difficulty
3. **Quality ceiling**: Combines the efficiency of small models with the quality of large ones
4. **Latency tradeoff**: More inference compute = higher latency (but can be parallelized)

### Interview-Ready Formulation

"Test-time compute scaling represents a paradigm shift in how we think about model deployment. Rather than a single forward pass, we treat inference as a search process. The key insight is that for tasks with verifiable answers, generating more candidates and selecting the best one is often more compute-efficient than training a larger model. The practical challenge is building good verifiers and managing latency."

In [ ]:
# Demonstrate test-time compute scaling with best-of-N

@torch.no_grad()
def best_of_n(model, tokenizer, prompt, gold_answer, N_values=[1, 2, 4, 8, 16]):
    """Demonstrate how accuracy improves with more samples at test time."""
    results = {}
    
    for N in N_values:
        responses = []
        for _ in range(N):
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)
            output = model.generate(
                **inputs, max_new_tokens=100, do_sample=True,
                temperature=0.8, top_p=0.95, pad_token_id=tokenizer.pad_token_id
            )
            resp = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            responses.append(resp)
        
        # Check if ANY of the N responses is correct
        any_correct = False
        for resp in responses:
            extracted = GRPO._extract_answer(resp)
            if extracted and GRPO._answers_match(extracted, gold_answer):
                any_correct = True
                break
        
        results[N] = any_correct
    
    return results


# Test on evaluation questions
print("Demonstrating test-time compute scaling (best-of-N)...")
N_VALUES = [1, 2, 4, 8]
n_test = min(20, len(eval_questions))
accuracy_by_N = {N: 0 for N in N_VALUES}

model.eval()
for i in range(n_test):
    if i % 5 == 0:
        print(f"  Testing question {i+1}/{n_test}...")
    prompt = format_math_prompt(eval_questions[i])
    results = best_of_n(model, tokenizer, prompt, eval_answers[i], N_VALUES)
    for N, correct in results.items():
        accuracy_by_N[N] += int(correct)

print("\nTest-Time Compute Scaling Results:")
print("-" * 40)
for N in N_VALUES:
    acc = accuracy_by_N[N] / n_test
    print(f"  Best-of-{N:2d}: {acc:.1%} ({accuracy_by_N[N]}/{n_test})")

In [ ]:
# Plot test-time compute scaling
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

accuracies = [accuracy_by_N[N] / n_test for N in N_VALUES]
ax.plot(N_VALUES, accuracies, 'o-', color="#1f77b4", linewidth=2.5, markersize=10)
ax.fill_between(N_VALUES, accuracies, alpha=0.1, color="#1f77b4")
ax.set_xlabel("Number of Samples (N)", fontsize=12)
ax.set_ylabel("Accuracy (best-of-N)", fontsize=12)
ax.set_title("Test-Time Compute Scaling: Best-of-N on GSM8K", fontsize=14, fontweight="bold")
ax.set_ylim(-0.05, 1.05)
ax.set_xscale("log", base=2)
ax.set_xticks(N_VALUES)
ax.set_xticklabels([str(n) for n in N_VALUES])

# Annotate
for N, acc in zip(N_VALUES, accuracies):
    ax.annotate(f"{acc:.0%}", (N, acc), textcoords="offset points", 
                xytext=(0, 12), ha="center", fontsize=11, fontweight="bold")

ax.axhline(y=accuracies[0], color="gray", linestyle="--", alpha=0.5, label=f"Single sample: {accuracies[0]:.0%}")
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("test_time_scaling.png", dpi=150, bbox_inches="tight")
plt.show()
print("Test-time scaling plot saved.")

---
## 10. "Why Does This Work?" -- Deep Understanding Prompts

### "Why does RL produce reasoning that SFT cannot?"

SFT trains the model to **imitate** reasoning traces. The model learns the surface pattern of step-by-step solutions. But:
- It does not learn **why** each step is useful
- It does not learn to **adapt** its reasoning strategy to novel problems
- It copies the reasoning style of the training data, including its mistakes

RL trains the model to **maximize reward** (correctness). The model discovers through trial-and-error that:
- Breaking problems into steps leads to more correct answers
- Checking intermediate results catches errors
- Showing work helps the model's own "internal state" track the computation

The key difference: SFT learns **what reasoning looks like**. RL learns **what reasoning is for**.

### "Why is GRPO simpler than PPO but still works?"

The critic (value function) in PPO serves as a **baseline** to reduce variance in the policy gradient. GRPO replaces this with a simpler baseline: the **group average**.

Why this is sufficient:
1. For language generation tasks, the reward is computed per-episode (not per-token), so you do not need a fine-grained value estimate
2. The group average is an unbiased estimator of the expected reward under the current policy
3. With G=8-16 samples, the variance of this estimator is manageable
4. The learned critic in PPO can introduce its own biases and training instabilities

When GRPO might struggle: if the reward function is very sparse (most completions get 0 reward), the group normalization becomes noisy.

### "What are the limits of verifiable reward domains?"

Verifiable rewards work well for:
- **Math**: answer can be checked
- **Code**: tests can be run
- **Logic puzzles**: solution can be verified
- **Factual QA**: answer can be looked up

They do **not** work well for:
- **Creative writing**: quality is subjective
- **Nuanced ethical reasoning**: no clear "correct" answer
- **Open-ended advice**: multiple valid approaches
- **Summarization**: quality has multiple dimensions

For non-verifiable domains, you still need learned reward models (RLHF) or constitutional approaches.

### "Can this approach work for non-math/non-code tasks?"

Potentially, but with caveats:

1. **With synthetic verification**: Use a strong model (GPT-4, Claude) as a judge to provide reward signals. This is "RL from AI Feedback" (RLAIF).
2. **With constrained tasks**: Any task with verifiable constraints (e.g., "write a poem that rhymes and has 14 lines") can use rule-based rewards.
3. **With human preferences**: Use RLHF-style rewards but with the GRPO algorithm (replace PPO with GRPO in the standard pipeline).

The DeepSeek team showed that GRPO works with learned reward models too -- it is not limited to verifiable rewards. The verifiable reward setup is just the cleanest demonstration of the paradigm.

**Insider Tip:** Reasoning via RL is the hottest research area right now. If you're interviewing at OpenAI or DeepSeek, expect deep questions on: (1) How does RL produce reasoning that SFT can't? (2) What's the role of verifiable rewards? (3) How do you handle non-verifiable domains? Have opinions.

**Insider Tip:** Process Reward Models (PRMs) are likely a key ingredient in o1/o3. The "Let's Verify Step by Step" paper (Lightman et al. 2023) is essential reading. In an interview, mentioning PRMs vs ORMs and the trade-offs shows you understand the frontier.

**Reference:** See also the "Qwen2.5-Math Technical Report" (2024) for additional details on GRPO applied to math reasoning at scale. It provides valuable complementary evidence that GRPO scales well and offers practical implementation insights beyond what DeepSeekMath covers.

---
## Interview Question Bank: Reasoning Models & GRPO (HIGHEST PRIORITY)

*This is THE interview topic for 2025-2026. If you are interviewing at OpenAI, Anthropic, DeepSeek, Google DeepMind, or any frontier lab, you WILL be asked about reasoning via RL. The questions below are ordered from most to least likely. Prepare Q1 and Q2 first.*

---

**Q1: "How does DeepSeek-R1 train reasoning without any chain-of-thought supervision?"**

**What we're testing:** Whether you are current on the most important development in post-training since RLHF. This is THE question for 2025-2026. If you cannot answer this clearly, you will not be seen as current enough for a Senior ML researcher role.

**Good answer:**
- DeepSeek-R1-**Zero** uses reinforcement learning with correctness rewards to train reasoning, with no supervised chain-of-thought data. Important: that is true only of R1-Zero. The released DeepSeek-R1 model used a cold-start CoT SFT phase followed by multiple stages of SFT and RL.
- The model is given math/code problems, generates solutions, and receives a binary reward: 1 if the final answer is correct, 0 otherwise.
- Reasoning (chain-of-thought, self-verification, backtracking) emerges as the model discovers that these behaviors increase its reward.
- The RL algorithm used is GRPO (Group Relative Policy Optimization), not PPO.

**Great answer (Senior -> Principal level):**
- Explains GRPO specifically: "For each prompt, sample a group of G completions. Compute the reward for each. Normalize rewards within the group (subtract mean, divide by std). Use the normalized rewards as advantages in a clipped surrogate policy gradient loss. This eliminates the need for a learned value function (critic), which is the main instability source in PPO for LLMs."
- Distinguishes R1-Zero vs R1: "R1-Zero trains purely with RL from the base model -- no SFT. This produces reasoning but with poor formatting and readability. R1 adds a cold-start SFT phase: first train on a small amount of curated long-CoT data -- largely model-generated (e.g., few-shot prompting and refined, cleaned-up R1-Zero outputs, with human post-processing), not human-written from scratch -- THEN apply GRPO, followed by further SFT and RL stages. This dramatically improves output quality while preserving the emergent reasoning."
- Discusses the reward design: "Two rewards are used simultaneously: (1) correctness reward (binary, verifiable) and (2) format reward -- in R1-Zero this enforces wrapping the reasoning process in `<think></think>` tags (answer-extraction formats like \\boxed{} belong to the correctness checking for math, not the format reward). The format reward is crucial -- without it, the model produces correct but unparseable outputs."
- Explains the distillation story: "After training R1 (671B MoE), DeepSeek distilled its reasoning traces into smaller dense models (1.5B to 70B). The distilled models often outperform the same-size models trained directly with RL, suggesting that reasoning is easier to distill than to discover."
- Connects to the broader paradigm: "This is the shift from 'reasoning via prompting' (chain-of-thought prompting, 2022) to 'reasoning via training' (RL on verifiable tasks, 2024-2025). The model does not learn to mimic reasoning -- it discovers reasoning as an instrumental strategy for getting rewards."

**Red flag:** Cannot explain GRPO. Confuses R1 with chain-of-thought prompting. Says "they just trained on reasoning data." Does not know the difference between R1-Zero and R1.

**Follow-up:** "Can this approach work for tasks WITHOUT verifiable rewards? How?"
- Expected: This is an open research problem. Approaches: (1) use a reward model trained on human preferences as a proxy for correctness, (2) use constitutional AI-style self-critique as a reward signal, (3) use process reward models that score individual reasoning steps. A great candidate acknowledges the fundamental limitation: "Verifiable rewards are what make reasoning RL work so well. For non-verifiable domains (creative writing, open-ended advice), we either need much better reward models or a fundamentally different approach."

**Follow-up:** "What is the role of the format reward in R1 training?"
- Expected: The format reward ensures the model produces structured outputs (in R1-Zero: the reasoning process wrapped in `<think></think>` tags, followed by the answer). Without it, the model may discover correct reasoning but express it in ways that are unparseable or unreadable. It is a pragmatic engineering choice that turns out to be important for both evaluation and user experience.

---

**Q2: "Implement GRPO from scratch on a whiteboard."**

**What we're testing:** Deep algorithmic understanding, coding fluency, ability to translate math to code under pressure.

**Context:** Expected at DeepSeek, likely at OpenAI/Anthropic in 2025-2026. 20 minutes. This is a coding question, not just conceptual.

**Key components that MUST appear:**
1. Sample a group of G completions for each prompt
2. Compute rewards for each completion (binary correctness or reward model score)
3. Normalize rewards within each group: advantage = (reward - mean) / std
4. Compute clipped surrogate loss (PPO-style): min(ratio * advantage, clip(ratio, 1-eps, 1+eps) * advantage)
5. Add KL penalty term: -beta * KL(pi || pi_ref)
6. Update policy with gradient descent

**Good answer (pseudocode):**
```python
for prompt in batch:
    # 1. Sample group
    completions = [model.generate(prompt) for _ in range(G)]
    # 2. Compute rewards
    rewards = [reward_fn(prompt, c) for c in completions]
    # 3. Normalize within group
    advantages = (rewards - mean(rewards)) / (std(rewards) + eps)
    # 4. Compute log ratios
    for completion, advantage in zip(completions, advantages):
        log_ratio = logprob(model, completion) - logprob(old_model, completion)
        ratio = exp(log_ratio)
        # 5. Clipped surrogate
        loss1 = ratio * advantage
        loss2 = clip(ratio, 1-eps, 1+eps) * advantage
        policy_loss = -min(loss1, loss2)
        # 6. KL penalty
        kl = logprob(model, completion) - logprob(ref_model, completion)
        total_loss = policy_loss + beta * kl
    optimizer.step()
```

**Great answer:** All of the above PLUS:
- Handles the per-token vs per-sequence distinction: log probabilities are summed over tokens
- Discusses why group normalization works: "It replaces the learned value baseline in PPO. Each prompt's group provides its own local baseline. This is closely related to -- but NOT equivalent to -- REINFORCE with a leave-one-out baseline (RLOO): GRPO includes the sample itself in the group mean and divides by the group std, whereas RLOO excludes the sample from its baseline and does not std-normalize."
- Notes the memory optimization: "GRPO eliminates the value network entirely. In PPO for LLMs, the value network is a full copy of the policy -- removing it saves 33-50% memory."

**Red flag:** Cannot write any code. Confuses GRPO with standard REINFORCE. Does not include the clipping step. Does not know about group normalization.

**Follow-up:** "How is GRPO different from REINFORCE with a baseline?"
- Expected: GRPO uses the group mean as a baseline (no learned baseline), applies PPO-style clipping for stability, and operates on groups of completions per prompt rather than single samples. The clipping is the key stability improvement over vanilla REINFORCE.

---

**Q3: "Process reward models vs outcome reward models -- which is better and why?"**

**What we're testing:** Knowledge of the reward model landscape for reasoning, ability to reason about supervision granularity trade-offs.

**Good answer:**
- **ORMs (Outcome Reward Models):** Score the final answer only. Easy to train (just need answer correctness). Used in DeepSeek-R1 (binary correctness reward).
- **PRMs (Process Reward Models):** Score each reasoning step. More informative signal. Referenced in "Let's Verify Step by Step" (Lightman et al., 2023).
- PRMs provide denser supervision (reward at every step vs only at the end), which should accelerate learning. ORMs are cheaper to train (only need final answer labels, not step-level annotations).

**Great answer (Senior -> Principal level):**
- Discusses the "Let's Verify Step by Step" paper: "OpenAI showed that PRMs significantly outperform ORMs for best-of-N selection on math problems. The PRM can identify WHERE reasoning goes wrong, not just WHETHER it went wrong."
- Addresses the labeling bottleneck: "The main problem with PRMs is that step-level labels are extremely expensive. A human annotator needs to verify each reasoning step, which takes 10-50x longer than verifying a final answer."
- Discusses automated PRM training: "Recent work (Math-Shepherd, OmegaPRM) shows that you can train PRMs automatically using Monte Carlo rollouts: for each intermediate step, generate many completions from that point and estimate the step's quality by the fraction that reach the correct answer. This eliminates the need for human step-level labels."
- Knows when ORMs are sufficient: "For GRPO-style training where you just need a binary signal, ORMs work well because the group normalization provides implicit credit assignment. PRMs are most valuable for test-time search (beam search with step-level scoring) and for debugging reasoning failures."
- Connects to test-time compute: "PRMs enable more efficient test-time compute scaling because you can prune bad reasoning paths early, rather than generating entire solutions and checking the answer."

**Red flag:** Does not know what a PRM is. Cannot explain the labeling cost difference. Says "PRMs are strictly better" without discussing the cost trade-off.

**Follow-up:** "How would you train a PRM without human step-level labels?"
- Expected: Monte Carlo Tree Search-style estimation: at each reasoning step, sample N completions to the end, compute the fraction that reach the correct answer. That fraction becomes the step-level reward label. This is the approach used in Math-Shepherd and adopted by several frontier labs.

---

**Q4: "Explain test-time compute scaling. When is it more efficient than model scaling?"**

**What we're testing:** Understanding of the new scaling paradigm, ability to reason about compute allocation, knowledge of the Snell et al. results.

**Good answer:**
- Test-time compute scaling: instead of training a bigger model, spend more compute at inference time by generating multiple solutions and selecting the best one.
- Methods: best-of-N sampling, beam search with PRMs, iterative refinement, tree search with self-evaluation.
- More compute at inference = better accuracy, especially on reasoning tasks.

**Great answer (Senior -> Principal level):**
- Discusses optimal compute allocation (Snell et al., 2024): "There is a crossover point where spending X FLOPs on test-time compute gives more accuracy improvement than spending X FLOPs on additional pretraining. For difficult problems, test-time scaling is more efficient. For easy problems, model scaling wins."
- Quantifies the trade-off: "A 7B model with best-of-256 sampling can match or exceed a 70B model on math benchmarks, at roughly similar total FLOPs. But the 7B model needs 256x the inference time per query."
- Discusses the practical implications: "This creates a new product design dimension: easy queries get 1 sample (fast, cheap), hard queries get 64-256 samples (slow, expensive). You need difficulty estimation to route queries efficiently."
- Connects to reasoning models: "DeepSeek-R1 and OpenAI o1/o3 are essentially doing test-time compute scaling INSIDE the model -- the long chain-of-thought is an internal search process. The model has learned to allocate variable compute per problem by varying its reasoning length."
- Notes the limitations: "Test-time scaling works best when you have a reliable verifier (PRM or ground-truth checker). Without verification, generating more samples may not help -- you are just sampling from the same distribution."

**Red flag:** Only mentions "generate more and pick the best." Cannot explain when test-time scaling is more efficient than model scaling. Does not know about verification/scoring.

---
## Production Implementation Notes: Reasoning Models

### DeepSeek-R1 Training at Scale

What the paper describes vs what actually happens:

- **Compute**: large-scale GRPO training (DeepSeek did not publish a precise GPU count for R1's RL phase). The generation phase (sampling G completions per prompt) is the bottleneck -- it is embarrassingly parallel but requires massive inference throughput.
- **Task distribution**: Math (GSM8K, MATH, competition problems) and code (Codeforces, LeetCode) dominate because they have automated verification. This is not a coincidence -- verifiable rewards are the prerequisite.
- **Reward design**: Binary correctness for math (answer matches ground truth), pass/fail for code (test cases), format compliance (regex matching for structured output). The simplicity of the reward is the feature, not a limitation.
- **Training dynamics**: The R1 paper describes response length and reasoning behaviors (self-correction, backtracking, verification -- the "aha moment") emerging gradually over the course of RL training. It describes this qualitatively; no public step counts mark when each behavior appears.
- **Scaling the group size G**: Larger groups provide better advantage estimates but cost more compute per step. For a sourced number: DeepSeekMath sampled 64 outputs per question for GRPO; the R1 paper does not publish per-domain group sizes.

### Serving Reasoning Models: The Latency Challenge

Reasoning models produce variable-length chain-of-thought outputs. A simple factual question might get 50 tokens; a hard math problem might get 5000+ tokens. This creates unique serving challenges:

- **Unpredictable latency**: Standard batching assumes roughly uniform output lengths. With reasoning models, one request in a batch might finish in 100ms while another takes 30 seconds. Solution: speculative decoding, continuous batching (vLLM), or chunked prefill.
- **KV cache management**: Long chain-of-thought outputs consume large KV caches. For Llama-70B with GQA (80 layers, 8 KV heads, head_dim 128, FP16), the KV cache costs 2 (K and V) x 80 x 8 x 128 x 2 bytes = ~0.33 MB per token, so a 4096-token request needs ~1.3 GB, and batch size 32 needs ~43 GB. (Even with full MHA, 64 heads, it would be ~11 GB per request, not 32 GB.) Solution: paged attention (vLLM), KV cache compression, or quantized KV cache.
- **Cost optimization**: Reasoning models are 10-100x more expensive per query than standard models (because of the long CoT). Production deployments must estimate query difficulty and route easy queries to cheaper models. This is an active engineering challenge.

### Evaluation: What Benchmarks Actually Matter

| Benchmark | What It Tests | Difficulty | Notes |
|-----------|--------------|------------|-------|
| GSM8K | Grade-school math | Easy (saturated) | R1 gets 97%+. Not useful for frontier models. |
| MATH | Competition math | Medium | R1 gets 79-97% depending on level. Still useful. |
| AIME 2024 | AMC/AIME competition | Hard | THE reasoning benchmark of the 2024-2025 era (see notebook 23 for the current landscape). R1: ~79% on AIME 2024. |
| Codeforces | Competitive programming | Hard | Measures both reasoning and coding. R1: ~96th percentile. |
| GPQA Diamond | PhD-level science | Very Hard | Tests multi-domain reasoning. Useful for generalization. |

**Non-verifiable domains remain an open challenge.** There is no good benchmark for reasoning about open-ended questions (ethics, strategy, creative problem-solving). This is an active research frontier.

---
## How Reasoning Models Get Tested in Interviews

### This Topic Is Non-Negotiable for 2025-2026 Interviews

If you are interviewing for any post-training role, you will be asked about reasoning models. The depth of questioning varies by company:

| Company | Depth Expected | Likely Questions |
|---------|---------------|------------------|
| **DeepSeek** | Exhaustive -- they built R1 | Implementation details of GRPO, training dynamics, scaling decisions, reward engineering |
| **OpenAI** | Deep -- they built o1/o3 | Conceptual: "How would YOU train a reasoning model?", followed by system design and trade-offs |
| **Anthropic** | Conceptual + safety | How reasoning models change the alignment landscape, safety implications of emergent behaviors |
| **Google DeepMind** | Broad + theoretical | Connections to planning, search, MCTS. "How does this relate to AlphaGo?" |
| **Meta (FAIR)** | Research-oriented | "What would you do differently than R1?" -- wants novel ideas |

### The 4-Week Prep Plan for Reasoning Model Questions

**Week 1: Read the Papers**
- DeepSeek-R1 paper (the full paper, not a summary). Focus on Sections 3 (method) and 4 (experiments).
- "Let's Verify Step by Step" (Lightman et al., 2023). Understand PRM vs ORM.
- Snell et al. (2024) on test-time compute scaling.

**Week 2: Implement GRPO**
- Code GRPO from scratch (not using a library). Train on GSM8K.
- Reproduce the "emergent reasoning" result even at small scale (1B model).
- Time yourself: can you implement GRPO on a whiteboard in 20 minutes?

**Week 3: System Design**
- Design a complete reasoning model training pipeline. Write it out: data collection, reward design, training schedule, evaluation gates, deployment plan.
- Think about: "What if I had to train a reasoning model for a domain without verifiable answers?"
- Study the serving challenges: how would you deploy a model that generates 500-5000 tokens of reasoning per query?

**Week 4: Mock Interviews**
- Have someone ask you Q1 ("How does R1 train reasoning?"). Your answer should be 3-5 minutes, covering GRPO, R1-Zero vs R1, reward design, distillation, and the broader paradigm shift. Practice until it is fluent.
- Practice Q2 (implement GRPO) on paper. No IDE, no autocomplete.
- Practice transitioning from "here is how R1 works" to "here is what I would do differently" -- that transition is the Principal-level signal.

### The Question Behind Every Question

When an interviewer asks about reasoning models, they are really asking: "Do you understand the most important paradigm shift in post-training since RLHF? And can you contribute to pushing it further?"

Your answer should demonstrate:
1. You understand the current state-of-the-art (R1, o1/o3, GRPO)
2. You know the limitations (verifiable rewards only, high compute cost, non-verifiable domains unsolved)
3. You have ideas for what comes next (this is the Principal-level differentiator)

---
## 11. Flashcard Summary

| # | Question | Answer |
|---|----------|--------|
| 1 | What is the core idea of reasoning via RL? | Train the model to reason by giving it reward for correct answers, rather than imitating reasoning traces via SFT |
| 2 | What did DeepSeek-R1-Zero demonstrate? | Pure RL (no SFT on reasoning traces) produces emergent chain-of-thought, including self-correction and verification |
| 3 | What does GRPO stand for and what is its key innovation? | Group Relative Policy Optimization. It eliminates the critic network by using group-normalized rewards as advantages |
| 4 | How does GRPO compute advantages? | advantage_i = (r_i - mean(r_group)) / std(r_group), where r_group are rewards from G sampled completions |
| 5 | How much memory does GRPO save vs PPO? | ~50% (no critic/value network to store) |
| 6 | What is the tradeoff of GRPO vs PPO? | GRPO needs G samples per prompt (more inference), but avoids training a value network (less memory, simpler code) |
| 7 | What are verifiable rewards? Give examples. | Rewards that can be checked automatically: math answer correctness, code test passing, logic puzzle verification |
| 8 | What is the difference between ORM and PRM? | ORM rewards only the final answer. PRM rewards each intermediate reasoning step. PRMs are more effective but harder to annotate |
| 9 | What did "Let's Verify Step by Step" show? | PRMs significantly outperform ORMs, especially with best-of-N sampling. At best-of-1860 on MATH, the PRM reaches 78.2% vs 72.4% (ORM) and 69.6% (majority voting) |
| 10 | What is test-time compute scaling? | Instead of training a bigger model, spend more compute at inference: generate more candidates, use verifier to pick best |
| 11 | When is test-time scaling most effective? | For problems the model can solve ~10-50% of the time with a single attempt. Not useful for trivially easy or impossibly hard problems |
| 12 | Why does RL produce reasoning that SFT cannot? | SFT learns what reasoning looks like (imitation). RL learns what reasoning is for (maximizing correctness). RL discovers reasoning as a strategy |
| 13 | Can GRPO work with non-verifiable rewards? | Yes, GRPO can use learned reward models too. Verifiable rewards are the cleanest setup but not the only one |
| 14 | What is the DeepSeekMath paper's key contribution? | Introduced GRPO; its headline win is efficiency (no critic) at comparable quality on math reasoning -- the paper did not run a controlled PPO-vs-GRPO head-to-head |
| 15 | How does o1/o3 likely work (based on public information)? | RL training for reasoning + test-time compute scaling + likely PRMs for step-level guidance. Details are proprietary |

---
## 12. Paper Guides

### Paper 1: DeepSeek-R1
- **Title:** DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning
- **URL:** https://arxiv.org/abs/2501.12948
- **Why read it:** The definitive paper on reasoning via RL. Shows that pure RL produces emergent CoT, self-correction, and "aha moments" without any SFT on reasoning traces.
- **Key sections:** Section 2.2 (DeepSeek-R1-Zero -- the pure RL experiment), Section 2.3 (DeepSeek-R1 -- the full pipeline with cold-start SFT + RL), Section 2.4 (distillation to smaller models).
- **Interview relevance:** Very high. You will be asked about this paper at any frontier lab interview.

### Paper 2: DeepSeekMath / GRPO
- **Title:** DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models
- **URL:** https://arxiv.org/abs/2402.03300
- **Why read it:** Introduces GRPO, the algorithm that powers DeepSeek-R1. Clean and well-written.
- **Key sections:** Section 4.1 (the GRPO algorithm, including how it modifies PPO).
- **Interview relevance:** High. You should be able to explain GRPO vs PPO at the whiteboard.

### Paper 3: Let's Verify Step by Step
- **Title:** Let's Verify Step by Step
- **URL:** https://arxiv.org/abs/2305.20050
- **Why read it:** Establishes that process reward models (PRMs) outperform outcome reward models (ORMs) for math reasoning. The foundation for o1-style reasoning.
- **Key sections:** Section 2 (methods -- including the PRM800K data-collection process), Section 3 (large-scale PRM vs ORM comparison).
- **Interview relevance:** High. PRMs are central to the reasoning model paradigm.

### Paper 4: Scaling LLM Test-Time Compute Optimally
- **Title:** Scaling LLM Test-Time Compute Optimally Can be More Effective than Scaling Model Parameters
- **URL:** https://arxiv.org/abs/2408.03314
- **Why read it:** Rigorous analysis of when and how to scale test-time compute. Provides the theoretical framework for the new scaling axis.
- **Key sections:** Section 3 (compute-optimal allocation), Section 4 (when test-time scaling beats model scaling), Section 5 (adaptive strategies).
- **Interview relevance:** High. Understanding the test-time compute tradeoff is essential for reasoning model discussions.

In [ ]:
# Cleanup
del model, ref_model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
print("Notebook complete. GRPO training artifacts saved.")